In [1]:
import threading
import time
from pathlib import Path
from IPython.display import display, Markdown, update_display

from monitor_utils import sweep_quality

FITDATA = Path("carbonara_runs/smarcalAlpha/fitdata")

_monitor_stop = threading.Event()
_monitor_thread = None
_monitor_display_id = None


def start_monitor(threshold=2.5, every_s=10):
    global _monitor_thread, _monitor_display_id

    _monitor_stop.clear()

    # THIS creates the output area in the notebook
    handle = display(Markdown("Starting monitor..."), display_id=True)
    _monitor_display_id = handle.display_id

    def loop():
        while not _monitor_stop.is_set():
            stats = sweep_quality(FITDATA, threshold)

            lines = []
            lines.append("## 🔬 Carbonara Live Monitor")
            lines.append("")
            lines.append(f"**Threshold (χ²):** {threshold}")
            lines.append(f"**Good models:** {stats['good']} / {stats['total']}")
            lines.append(f"**Errors:** {stats['errors']}")
            if stats["best"] is not None:
                lines.append(f"**Best χ²:** {stats['best']:.4f}")

            update_display(
                Markdown("\n".join(lines)),
                display_id=_monitor_display_id
            )

            time.sleep(every_s)

    _monitor_thread = threading.Thread(target=loop, daemon=True)
    _monitor_thread.start()


def stop_monitor():
    _monitor_stop.set()

In [2]:
from run_frontend import CarbonaraRunner

runner = CarbonaraRunner()

In [3]:
runner.start()
start_monitor(threshold=2.5)

🚀 Carbonara started (PID=434033)


## 🔬 Carbonara Live Monitor

**Threshold (χ²):** 2.5
**Good models:** 98 / 131
**Errors:** 0
**Best χ²:** 0.5313

Clearing directory: /home/rdata/ktch24/carbonara2/carbonara_runs/smarcalAlpha/fitdata
Watcher started (PID=434055)
\n
 >> Run number : 1 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 2 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 3 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 4 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 5 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 6 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 7 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 8 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 9 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 10 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 11 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 12 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 13 
\n
Max number of fitting steps:  10000
\n
\n
 >> Run number : 14 
\n
Max numbe

In [ ]:
#To stop monitoring

stop_monitor()


In [ ]:
runner.stop()

<font size="6">  
    To analyse the predictions (this can be done whilst the fitting algorithm is running)
</font> 

In [4]:
import backmapping_funcs as bm
import CarbonaraDataTools as cdt
import fittingAnalysis as fa
import numpy as np
import os
import json
import pickle
import sys
sys.path.insert(0, "./_local_pkgs")

In [5]:
directory = "carbonara_runs/smarcalAlpha"
run = "fitdata"

<font size="6">  
    Collect all the good predictions, 
</font> 

In [56]:
good_preds = collect_good_prediction_files(
    fitdata_dir="carbonara_runs/smarcalAlpha/fitdata",
    chi2_threshold=2.5
)

print(f"Found {len(good_preds)} good predictions")
#for pdb, chi2 in good_preds[:len(good_preds)]:
 #   print(f"{chi2:.3f}  {pdb}")

Found 70 good predictions


In [63]:
good_files = [pdb for pdb, chi2 in good_preds]

Compare the variation in RMSD of predictions

In [64]:
rmsdPairedComparisons = fa.pairwise_structure_metrics(
    good_files,
    fa.compare_structures_vals
)

Pairwise comparisons:  45%|█████████████████▏                    | 1089/2415 [17:48<21:40,  1.02pair/s]


KeyboardInterrupt: 

In [ ]:
originalPdb = "pdbFiles/foldSmarcal.cif"

rmsdToOriginal = fa.structure_metrics_vs_carbonara(
    pdb_files=good_files,
    carbonara_dir=directory,
    compare_func=fa.compare_structures_vals_carbonara
)

In [ ]:
metric = "rmsd"
#could be 'tm' or 'gdt_ts'


fa.plot_overlaid_histograms([rmsdPairedComparisons,rmsdToOriginal],
    "rmsd",
    ["Carbonara-Carbonara","Carbonara-Original"],
    "RMSD (Angstroms)",
    "Comparative variations in RMSD Score of IL predictions",
    bins=40,
    figsize=(4.5, 3.5),
    colors=["blue","red"],
    alpha=0.5,
    density=False,
    dpi=300
)

# you can look in the variation in Radius of Gyration also

In [ ]:
# you can look in the variation in Radius of Gyration also

rg_results = fa.calc_rg_distribution(
    pdb_files=good_files,
    rg_func=fa.radius_of_gyration,
    weighted=False
)

# if its a mixture we can do the weighted Rg

#mixture_dict = {(runNo, "end"): mix for runNo, mix in zip(runNos, mixtures)}

#rg_results = calc_rg_distribution(
#    pdb_files=pdbs,
#    rg_func=fa.radius_of_gyration,
#    weighted=True,
#    mixtures=mixture_dict
#)

# lets do the og pdb
originalPdb = "pdbFiles/foldSmarcal.cif"

rg_results_og = fa.calc_rg_distribution(
    pdb_files=[originalPdb],
    rg_func=fa.radius_of_gyration,
    weighted=False
)

In [ ]:
metric = "rg"

fa.plot_overlaid_histograms([rg_results,rg_results_og],
    metric,
    ["carbonra predictions","original model"],
    "Rg (Angstroms)",
    "Variation in Rg of carbonara predictions compared to homology model",
    bins=40,
    figsize=(4.5, 3.5),
    colors=["blue","red"],
    alpha=0.5,
    density=False,
    dpi=300
)

You can also view the results if pymol3d installed 

In [40]:
fa.visualisePredictionIndividual(good_files[0])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [65]:
visualisePredictionComparison(good_files[0],good_files[11], do_superpose=True)

Superposed model 1 onto model 0 using 546 matched Cα atoms. RMSD = 15.912 Å


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [83]:
!{sys.executable} -m pipreqs.pipreqs "./" --force --scan-notebooks

Usage:
    pipreqs [options] [<path>]


In [86]:

import ast
import json
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()

# Standard library modules to ignore (not exhaustive, but enough for a useful first pass)
STDLIB = {
    "abc","argparse","ast","asyncio","base64","collections","concurrent","contextlib","copy",
    "csv","dataclasses","datetime","functools","glob","gzip","hashlib","heapq","html","importlib",
    "io","itertools","json","logging","math","multiprocessing","numbers","operator","os","pathlib",
    "pickle","platform","pprint","queue","random","re","shlex","shutil","signal","sqlite3","statistics",
    "string","subprocess","sys","tempfile","textwrap","threading","time","traceback","types","typing",
    "unittest","urllib","uuid","warnings","weakref","xml","zipfile"
}

# Local/internal modules you do NOT want in requirements
IGNORE_TOPLEVEL = {
    "monitor_utils",
    "run_frontend",
    "watch_and_backmap",
    "backmap_cli",
    "backmapping_funcs",
}

# Manual mapping from import name -> pip package name
NAME_MAP = {
    "Bio": "biopython",
    "IPython": "ipython",
    "PIL": "Pillow",
    "yaml": "PyYAML",
    "sklearn": "scikit-learn",
    "cv2": "opencv-python",
}

def top_imports_from_source(source: str):
    mods = set()
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return mods

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                mods.add(alias.name.split(".")[0])
        elif isinstance(node, ast.ImportFrom):
            if node.module:
                mods.add(node.module.split(".")[0])
    return mods

def imports_from_py(path: Path):
    try:
        return top_imports_from_source(path.read_text(encoding="utf-8"))
    except Exception:
        return set()

def imports_from_ipynb(path: Path):
    mods = set()
    try:
        nb = json.loads(path.read_text(encoding="utf-8"))
        for cell in nb.get("cells", []):
            if cell.get("cell_type") != "code":
                continue
            src = "".join(cell.get("source", []))
            mods |= top_imports_from_source(src)
    except Exception:
        pass
    return mods

all_mods = set()

for py in PROJECT_ROOT.rglob("*.py"):
    all_mods |= imports_from_py(py)

for nb in PROJECT_ROOT.rglob("*.ipynb"):
    all_mods |= imports_from_ipynb(nb)

# Clean up
mods = set()
for m in all_mods:
    if not m:
        continue
    if m in STDLIB:
        continue
    if m in IGNORE_TOPLEVEL:
        continue
    mods.add(NAME_MAP.get(m, m))

reqs = sorted(mods)

out_path = PROJECT_ROOT / "requirements.in"
out_path.write_text("\n".join(reqs) + "\n", encoding="utf-8")

print(f"Wrote {out_path}")
print("\n".join(reqs))

<unknown>:87: SyntaxWarning: invalid escape sequence '\c'
<unknown>:21: SyntaxWarning: invalid escape sequence '\s'
<unknown>:40: SyntaxWarning: invalid escape sequence '\s'
<unknown>:19: SyntaxWarning: invalid escape sequence '\s'
<unknown>:87: SyntaxWarning: invalid escape sequence '\c'
<unknown>:17: SyntaxWarning: invalid escape sequence '\s'
<unknown>:24: SyntaxWarning: invalid escape sequence '\#'
<unknown>:39: SyntaxWarning: invalid escape sequence '\s'
<unknown>:45: SyntaxWarning: invalid escape sequence '\s'


Wrote /home/rdata/ktch24/carbonara2/requirements.in
BME
BME_tools
CarbonaraDataTools
ChiScore
ConfigParser
Distribution
FitParameters
HTMLParser
IMP
MockGrid
OpenSSL
Pillow
Profile
Queue
RMF
RMF_HDF5
Residue
Scientific
Selector
StringIO
TuneRex
UserDict
_IMP_algebra
_IMP_atom
_IMP_benchmark
_IMP_cgal
_IMP_container
_IMP_core
_IMP_display
_IMP_domino
_IMP_em
_IMP_example
_IMP_foxs
_IMP_gsl
_IMP_isd
_IMP_kernel
_IMP_kinematics
_IMP_kmeans
_IMP_misc
_IMP_multi_state
_IMP_multifit
_IMP_npc
_IMP_pmi
_IMP_rmf
_IMP_rotamer
_IMP_saxs
_IMP_score_functor
_IMP_scratch
_IMP_statistics
_IMP_symmetry
_RMF
_RMF_HDF5
__builtin__
__future__
__main__
__pypy__
__version__
_abcoll
_aix_support
_api
_args
_async
_brokers
_callers
_cell_widths
_chrome_constants
_cli_utils
_cmsgpack
_code
_collections
_compat
_decorators
_dists
_elffile
_emoji_codes
_emoji_replace
_envs
_errors
_export_format
_extension
_fig_tools
_fileno
_frozen_importlib
_frozen_importlib_external
_hooks
_imp
_impl
_implementation
_in_proc